In [1]:
import warnings
warnings.filterwarnings('ignore')


(Cookbook_Editing_labels_and_structure_metadata)=
# Editing labels and structure metadata

*Updating native atom labels and querying selected B factors without changing atom indices.*

This recipe keeps labels, source indices, physical units and structure selections explicit and checks their H5MSM 0.5 persistence.

:::{versionadded} 1.0.0
:::


## Editing atom labels

Start from a fresh bundled molecular system and retain two atoms for a small control.


In [2]:
import molsysmt as msm
molsys = msm.convert(msm.systems['T4 lysozyme L99A']['181l.pdb'], to_form='molsysmt.MolSys')
molsys = msm.extract(molsys, selection=[0, 1])
msm.set(molsys, selection=[0], atom_id=101)
msm.set(molsys, selection=[1], atom_id='00101')
labels = msm.get(molsys, element='atom', atom_id=True)
print(labels)
assert labels == ['101', '00101']
assert msm.get(molsys, element='atom', atom_index=True) == [0, 1]


['101', '00101']


:::{note}
:class: dropdown

Explore the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` catalog for the bundled source. Atom IDs are labels: integers become strings, and `00101` is not equivalent to the label `101`.
:::


## Querying selected structures

Append two copies and assign distinct length-squared values. No dynamics calculation is required.


In [3]:
import molsysmt as msm
import numpy as np
from molsysmt import pyunitwizard as puw
second = msm.copy(molsys)
msm.append_structures(molsys, second)
msm.append_structures(molsys, second)
msm.set(molsys, b_factor=puw.quantity([[1, 2], [11, 12], [21, 22]], 'angstrom**2'))
factors = msm.get(molsys, structure_indices=[2, 0, 2], b_factor=True)
print(puw.get_value(factors, to_unit='angstrom**2'))
np.testing.assert_allclose(puw.get_value(factors, to_unit='angstrom**2'), [[21, 22], [1, 2], [21, 22]])
assert puw.get_value(msm.get(molsys, structure_indices=[], b_factor=True)).shape == (0, 2)


[[21. 22.]
 [ 1.  2.]
 [21. 22.]]


## Preserving labels and metadata

Save the system to H5MSM 0.5 and repeat the same public query after loading it.


In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as directory:
    path = str(Path(directory) / 'metadata.h5msm')
    msm.convert(molsys, to_form=path)
    restored = msm.convert(path, to_form='molsysmt.MolSys')
    assert msm.get(restored, element='atom', atom_id=True) == labels
    saved_factors = msm.get(restored, structure_indices=[2, 0, 2], b_factor=True)
    np.testing.assert_allclose(puw.get_value(saved_factors, to_unit='angstrom**2'), [[21, 22], [1, 2], [21, 22]])
print('String labels and selected B factors survive H5MSM 0.5.')


String labels and selected B factors survive H5MSM 0.5.


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Set`: editing attributes with {func}`molsysmt.basic.set`.
- {ref}`Tutorial_Get`: querying attributes with {func}`molsysmt.basic.get`.
- {ref}`Introduction_Attributes`: understanding the attribute axes and layers.
:::
